# CPU vs GPU Matrix Multiplication Lab

## Setup

In [ ]:
import statistics
import time
import torch

print('CUDA available:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')

In [ ]:
def benchmark(fn, repeats=10, warmups=3):
    for _ in range(warmups):
        fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        fn()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append(time.perf_counter() - start)
    return statistics.median(times)

## Theory-to-code bridge
Matrix multiplication scales well on parallel hardware due to repeated independent multiply-accumulate operations.

In [ ]:
sizes = [1024, 2048, 4096]
results = []

for size in sizes:
    a_cpu = torch.randn(size, size)
    b_cpu = torch.randn(size, size)
    cpu_time = benchmark(lambda: torch.matmul(a_cpu, b_cpu), repeats=5, warmups=2)

    gpu_time = None
    if torch.cuda.is_available():
        a_gpu = a_cpu.to('cuda')
        b_gpu = b_cpu.to('cuda')
        gpu_time = benchmark(lambda: torch.matmul(a_gpu, b_gpu), repeats=5, warmups=2)

    speedup = cpu_time / gpu_time if gpu_time else None
    results.append((size, cpu_time, gpu_time, speedup))

results

## Interpretation
Use the result table to identify size thresholds where GPU gains become meaningful.

## Extensions
- Repeat with FP16 where available
- Compare transfer-included vs transfer-excluded timings